In [ ]:

#24110019_Lê Minh Huy
"""
TÔ MÀU ĐỒ THỊ THỦ ĐỨC - INTERACTIVE PRO
"""

from __future__ import annotations

import argparse
import base64
import csv
import html
import math
import os
import sys
import textwrap
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import matplotlib

# Backend Agg chạy được trong môi trường headless như Jupyter/Colab.
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. DỮ LIỆU MẶC ĐỊNH
# ============================================================

DEFAULT_REGIONS = [
    "Linh Xuân",
    "Bình Chiểu",
    "Linh Trung",
    "Tam Bình",
    "Tam Phú",
    "Hiệp Bình Phước",
    "Hiệp Bình Chánh",
    "Linh Đông",
    "Linh Tây",
    "Linh Chiểu",
    "Trường Thọ",
    "Bình Thọ",
]

DEFAULT_EDGES = [
    ("Linh Xuân", "Bình Chiểu"),
    ("Linh Xuân", "Linh Trung"),
    ("Bình Chiểu", "Tam Bình"),
    ("Bình Chiểu", "Tam Phú"),
    ("Linh Trung", "Tam Phú"),
    ("Linh Trung", "Linh Tây"),
    ("Tam Bình", "Hiệp Bình Phước"),
    ("Tam Bình", "Tam Phú"),
    ("Tam Phú", "Hiệp Bình Chánh"),
    ("Tam Phú", "Linh Đông"),
    ("Hiệp Bình Phước", "Hiệp Bình Chánh"),
    ("Hiệp Bình Chánh", "Linh Đông"),
    ("Linh Đông", "Linh Tây"),
    ("Linh Tây", "Linh Chiểu"),
    ("Linh Chiểu", "Trường Thọ"),
    ("Trường Thọ", "Bình Thọ"),
    ("Bình Thọ", "Linh Đông"),
    ("Bình Thọ", "Linh Chiểu"),
]

# Vị trí vẽ chỉ để minh họa. Không phải tọa độ địa lý chính xác.
DEFAULT_POSITIONS = {
    "Linh Xuân": (0.2, 4.2),
    "Bình Chiểu": (2.1, 4.2),
    "Linh Trung": (0.9, 3.2),
    "Tam Bình": (3.6, 3.2),
    "Tam Phú": (2.1, 2.6),
    "Hiệp Bình Phước": (5.4, 3.2),
    "Hiệp Bình Chánh": (5.1, 2.0),
    "Linh Đông": (3.1, 1.6),
    "Linh Tây": (1.25, 1.6),
    "Linh Chiểu": (0.8, 0.55),
    "Trường Thọ": (2.45, 0.15),
    "Bình Thọ": (4.15, 0.55),
}

COLOR_PALETTES = {
    3: [
        ("Đỏ", "#ffb3b3"),
        ("Xanh", "#b9dcff"),
        ("Vàng", "#fff0a6"),
    ],
    4: [
        ("Đỏ", "#ffb3b3"),
        ("Xanh", "#b9dcff"),
        ("Vàng", "#fff0a6"),
        ("Tím", "#dfc8ff"),
    ],
    5: [
        ("Đỏ", "#ffb3b3"),
        ("Xanh", "#b9dcff"),
        ("Vàng", "#fff0a6"),
        ("Tím", "#dfc8ff"),
        ("Xanh lá", "#c9f7c5"),
    ],
}

ALGORITHM_LABELS = {
    "normal": "Backtracking - Theo danh sách",
    "degree": "Backtracking - Bậc lớn trước",
    "mrv": "Backtracking - MRV + Degree",
    "greedy": "Greedy - Bậc lớn trước",
}


# ============================================================
# 2. DATACLASS
# ============================================================

@dataclass
class TreeNode:
    node_id: int
    parent_id: Optional[int]
    label: str
    status: str
    depth: int


@dataclass
class RunResult:
    found: bool
    assignment: Dict[str, str]
    tree_nodes: List[TreeNode]
    logs: List[str]
    output_dir: Path
    graph_image: Path
    tree_image: Path
    csv_file: Path
    html_file: Path
    stats: Dict[str, int]


# ============================================================
# 3. PARSE DỮ LIỆU CẠNH
# ============================================================

def edges_to_text(edges: List[Tuple[str, str]]) -> str:
    return "\n".join(f"{a} - {b}" for a, b in edges)


def parse_edges_text(raw_text: str) -> Tuple[List[str], List[Tuple[str, str]]]:
    """
    Nhận text dạng:
        Linh Xuân - Bình Chiểu
        Linh Xuân, Linh Trung

    Trả về:
        regions, edges
    """
    edges: List[Tuple[str, str]] = []
    regions: List[str] = []

    for line_no, raw_line in enumerate(raw_text.splitlines(), start=1):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        if " - " in line:
            parts = line.split(" - ", 1)
        elif "," in line:
            parts = line.split(",", 1)
        elif "-" in line:
            parts = line.split("-", 1)
        else:
            raise ValueError(
                f"Dòng {line_no} sai định dạng: {raw_line}\n"
                "Dùng dạng: Khu vực A - Khu vực B"
            )

        a = parts[0].strip()
        b = parts[1].strip()

        if not a or not b:
            raise ValueError(f"Dòng {line_no} thiếu tên khu vực.")

        if a == b:
            raise ValueError(f"Dòng {line_no}: một khu vực không thể kề chính nó.")

        edge = (a, b)
        reverse = (b, a)
        if edge not in edges and reverse not in edges:
            edges.append(edge)

        if a not in regions:
            regions.append(a)
        if b not in regions:
            regions.append(b)

    if not edges:
        raise ValueError("Danh sách cạnh đang rỗng.")

    return regions, edges


# ============================================================
# 4. SOLVER
# ============================================================

class GraphColoringSolver:
    def __init__(
        self,
        regions: List[str],
        edges: List[Tuple[str, str]],
        color_names: List[str],
        algorithm: str,
    ) -> None:
        self.regions = regions
        self.edges = edges
        self.color_names = color_names
        self.algorithm = algorithm
        self.graph = self._build_graph()

        self.assignment: Dict[str, str] = {}
        self.tree_nodes: List[TreeNode] = []
        self.logs: List[str] = []

        self.try_count = 0
        self.violation_count = 0
        self.backtrack_count = 0

    def _build_graph(self) -> Dict[str, List[str]]:
        graph: Dict[str, List[str]] = defaultdict(list)

        for a, b in self.edges:
            if b not in graph[a]:
                graph[a].append(b)
            if a not in graph[b]:
                graph[b].append(a)

        for region in self.regions:
            graph.setdefault(region, [])

        return dict(graph)

    def degree(self, region: str) -> int:
        return len(self.graph.get(region, []))

    def add_tree_node(
        self,
        parent_id: Optional[int],
        label: str,
        status: str,
        depth: int,
    ) -> int:
        node_id = len(self.tree_nodes)
        self.tree_nodes.append(
            TreeNode(
                node_id=node_id,
                parent_id=parent_id,
                label=label,
                status=status,
                depth=depth,
            )
        )
        return node_id

    def is_safe(self, region: str, color: str) -> Tuple[bool, str]:
        for neighbor in self.graph[region]:
            if self.assignment.get(neighbor) == color:
                return False, f"{neighbor} đã tô {color}"
        return True, "Hợp lệ"

    def available_colors(self, region: str) -> List[str]:
        result = []
        for color in self.color_names:
            ok, _ = self.is_safe(region, color)
            if ok:
                result.append(color)
        return result

    def select_region(self) -> Optional[str]:
        unassigned = [r for r in self.regions if r not in self.assignment]

        if not unassigned:
            return None

        if self.algorithm == "normal":
            return unassigned[0]

        if self.algorithm in {"degree", "greedy"}:
            return max(unassigned, key=lambda r: self.degree(r))

        # MRV + Degree:
        # Ưu tiên vùng có ít màu hợp lệ nhất.
        # Nếu hòa, chọn vùng có bậc lớn hơn.
        return min(
            unassigned,
            key=lambda r: (len(self.available_colors(r)), -self.degree(r)),
        )

    def solve(self) -> bool:
        self.assignment.clear()
        self.tree_nodes.clear()
        self.logs.clear()
        self.try_count = 0
        self.violation_count = 0
        self.backtrack_count = 0

        if self.algorithm == "greedy":
            return self.solve_greedy()

        root_id = self.add_tree_node(None, "Assignment = {}", "root", 0)

        self.logs.append("Bắt đầu thuật toán Backtracking.")
        self.logs.append(f"Thuật toán: {ALGORITHM_LABELS.get(self.algorithm, self.algorithm)}")
        self.logs.append(f"Số vùng: {len(self.regions)}")
        self.logs.append(f"Số cạnh: {len(self.edges)}")
        self.logs.append(f"Màu: {', '.join(self.color_names)}")
        self.logs.append("-" * 70)

        found = self._backtrack(parent_id=root_id, depth=0)

        self.logs.append("-" * 70)
        self.logs.append("KẾT LUẬN: " + ("Tìm được nghiệm hợp lệ." if found else "Không tìm được nghiệm."))

        return found

    def _backtrack(self, parent_id: int, depth: int) -> bool:
        if len(self.assignment) == len(self.regions):
            self.add_tree_node(
                parent_id,
                "HOÀN THÀNH\nTìm được nghiệm",
                "solution",
                depth + 1,
            )
            return True

        region = self.select_region()
        if region is None:
            return False

        self.logs.append(f"Chọn vùng: {region}")

        for color in self.color_names:
            self.try_count += 1
            ok, reason = self.is_safe(region, color)

            if ok:
                self.assignment[region] = color
                child_id = self.add_tree_node(
                    parent_id,
                    f"{region} = {color}\nOK",
                    "ok",
                    depth + 1,
                )
                self.logs.append(f"  Thử {region} = {color}: OK")

                if self._backtrack(child_id, depth + 1):
                    return True

                self.backtrack_count += 1
                self.logs.append(f"  Quay lui: bỏ {region} = {color}")

                self.tree_nodes[child_id].status = "backtrack"
                self.tree_nodes[child_id].label = f"{region} = {color}\nQuay lui"
                del self.assignment[region]
            else:
                self.violation_count += 1
                self.add_tree_node(
                    parent_id,
                    f"{region} = {color}\nVi phạm\n{reason}",
                    "violation",
                    depth + 1,
                )
                self.logs.append(f"  Thử {region} = {color}: VI PHẠM ({reason})")

        return False

    def solve_greedy(self) -> bool:
        root_id = self.add_tree_node(None, "Greedy Coloring", "root", 0)
        parent_id = root_id

        order = sorted(self.regions, key=lambda r: self.degree(r), reverse=True)

        self.logs.append("Bắt đầu thuật toán Greedy.")
        self.logs.append(f"Thứ tự tô: {' -> '.join(order)}")
        self.logs.append(f"Màu: {', '.join(self.color_names)}")
        self.logs.append("-" * 70)

        for depth, region in enumerate(order, start=1):
            assigned = False

            for color in self.color_names:
                self.try_count += 1
                ok, reason = self.is_safe(region, color)

                if ok:
                    self.assignment[region] = color
                    parent_id = self.add_tree_node(
                        parent_id,
                        f"{region} = {color}\nOK",
                        "ok",
                        depth,
                    )
                    self.logs.append(f"Gán {region} = {color}: OK")
                    assigned = True
                    break

                self.violation_count += 1
                self.add_tree_node(
                    parent_id,
                    f"{region} = {color}\nVi phạm\n{reason}",
                    "violation",
                    depth,
                )
                self.logs.append(f"Thử {region} = {color}: VI PHẠM ({reason})")

            if not assigned:
                self.logs.append(f"Greedy thất bại tại vùng {region}.")
                return False

        self.add_tree_node(
            parent_id,
            "HOÀN THÀNH\nGreedy tìm được nghiệm",
            "solution",
            len(order) + 1,
        )

        self.logs.append("-" * 70)
        self.logs.append("KẾT LUẬN: Greedy tìm được nghiệm hợp lệ.")
        return True

    def validate_solution(self) -> Tuple[bool, List[str]]:
        errors = []

        for region in self.regions:
            if region not in self.assignment:
                errors.append(f"{region} chưa được tô màu.")

        for a, b in self.edges:
            if self.assignment.get(a) == self.assignment.get(b):
                errors.append(f"{a} và {b} cùng màu {self.assignment.get(a)}.")

        return len(errors) == 0, errors


# ============================================================
# 5. VẼ HÌNH
# ============================================================

def ensure_positions(regions: List[str]) -> Dict[str, Tuple[float, float]]:
    positions = dict(DEFAULT_POSITIONS)
    missing = [r for r in regions if r not in positions]

    if not missing:
        return positions

    radius = 2.7
    center_x = 3.0
    center_y = 2.0
    total = len(missing)

    for i, region in enumerate(missing):
        angle = 2 * math.pi * i / max(total, 1)
        positions[region] = (
            center_x + radius * math.cos(angle),
            center_y + radius * math.sin(angle),
        )

    return positions


def get_color_hex(color_name: str, color_count: int) -> str:
    for name, color_hex in COLOR_PALETTES[color_count]:
        if name == color_name:
            return color_hex
    return "#ffffff"


def draw_colored_graph(
    solver: GraphColoringSolver,
    color_count: int,
    output_path: Path,
) -> None:
    positions = ensure_positions(solver.regions)
    fig, ax = plt.subplots(figsize=(13, 8))

    for a, b in solver.edges:
        x1, y1 = positions[a]
        x2, y2 = positions[b]
        ax.plot([x1, x2], [y1, y2], color="#555555", linewidth=1.4, zorder=1)

    for region in solver.regions:
        x, y = positions[region]
        color_name = solver.assignment.get(region, "Chưa tô")
        face_color = get_color_hex(color_name, color_count)

        ax.scatter(
            x,
            y,
            s=2300,
            color=face_color,
            edgecolors="#222222",
            linewidths=1.7,
            zorder=3,
        )

        label = f"{region}\n{color_name}\nkề: {solver.degree(region)}"
        ax.text(x, y, label, ha="center", va="center", fontsize=8.3, zorder=4)

    ax.set_title(
        "Kết quả tô màu đồ thị khu vực Thủ Đức",
        fontsize=15,
        fontweight="bold",
    )
    ax.axis("off")
    ax.margins(0.16)
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def shorten_label(text: str, width: int = 18, max_lines: int = 4) -> str:
    lines: List[str] = []

    for part in text.splitlines():
        wrapped = textwrap.wrap(part, width=width) or [part]
        lines.extend(wrapped)

    if len(lines) > max_lines:
        lines = lines[: max_lines - 1] + ["..."]

    return "\n".join(lines)


def draw_backtracking_tree(
    tree_nodes: List[TreeNode],
    output_path: Path,
    max_nodes: int,
) -> None:
    if not tree_nodes:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.text(0.5, 0.5, "Chưa có dữ liệu cây", ha="center", va="center")
        ax.axis("off")
        fig.savefig(output_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        return

    visible_nodes = tree_nodes[:max_nodes]
    visible_ids = {node.node_id for node in visible_nodes}

    children: Dict[int, List[int]] = defaultdict(list)
    for node in visible_nodes:
        if node.parent_id is not None and node.parent_id in visible_ids:
            children[node.parent_id].append(node.node_id)

    root_id = visible_nodes[0].node_id
    positions: Dict[int, Tuple[float, float]] = {}
    next_x = [0.0]

    def layout(node_id: int, depth: int) -> float:
        child_list = children.get(node_id, [])

        if not child_list:
            x = next_x[0]
            next_x[0] += 1.0
        else:
            child_xs = [layout(child, depth + 1) for child in child_list]
            x = sum(child_xs) / len(child_xs)

        positions[node_id] = (x, -depth)
        return x

    layout(root_id, 0)

    width = max(14, min(34, next_x[0] * 0.65))
    height = max(8, min(26, max(node.depth for node in visible_nodes) * 1.3 + 2))

    fig, ax = plt.subplots(figsize=(width, height))

    for node in visible_nodes:
        if node.parent_id is None:
            continue
        if node.parent_id not in positions:
            continue

        x1, y1 = positions[node.parent_id]
        x2, y2 = positions[node.node_id]
        ax.plot([x1, x2], [y1, y2], color="#666666", linewidth=1.0, zorder=1)

    status_style = {
        "root": ("#f2f2f2", "#222222"),
        "ok": ("#d9fdd3", "#16831a"),
        "violation": ("#ffd6d6", "#b00020"),
        "backtrack": ("#eeeeee", "#777777"),
        "solution": ("#cce5ff", "#0057b8"),
    }

    for node in visible_nodes:
        x, y = positions[node.node_id]
        face_color, edge_color = status_style.get(node.status, ("#ffffff", "#222222"))

        ax.scatter(
            x,
            y,
            s=1850,
            color=face_color,
            edgecolors=edge_color,
            linewidths=1.5,
            zorder=3,
        )

        ax.text(
            x,
            y,
            shorten_label(node.label),
            ha="center",
            va="center",
            fontsize=7,
            zorder=4,
        )

    title = "Cây Backtracking / quá trình thử màu"
    if len(tree_nodes) > max_nodes:
        title += f" - hiển thị {max_nodes}/{len(tree_nodes)} node"

    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.axis("off")
    ax.margins(0.08)
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 6. XUẤT CSV + HTML
# ============================================================

def image_to_base64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode("utf-8")


def export_csv(solver: GraphColoringSolver, output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Khu vực", "Màu", "Số vùng kề", "Các vùng kề"])

        for region in solver.regions:
            writer.writerow(
                [
                    region,
                    solver.assignment.get(region, ""),
                    solver.degree(region),
                    ", ".join(solver.graph[region]),
                ]
            )


def result_table_html(solver: GraphColoringSolver) -> str:
    rows = []
    for region in solver.regions:
        rows.append(
            f"""
            <tr style="background:#ffffff !important; color:#111827 !important; opacity:1 !important;">
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{html.escape(region)}</td>
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">
                    <span style="display:inline-block; background:#dbeafe !important; color:#003fc7 !important; border-radius:999px; padding:4px 12px; font-weight:900 !important; opacity:1 !important;">
                        {html.escape(solver.assignment.get(region, ""))}
                    </span>
                </td>
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{solver.degree(region)}</td>
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{html.escape(", ".join(solver.graph[region]))}</td>
            </tr>
            """
        )
    return "\n".join(rows)


def edges_table_html(solver: GraphColoringSolver) -> str:
    rows = []
    for index, (a, b) in enumerate(solver.edges, start=1):
        rows.append(
            f"""
            <tr style="background:#ffffff !important; color:#111827 !important; opacity:1 !important;">
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{index}</td>
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{html.escape(a)}</td>
                <td style="color:#111827 !important; background:#ffffff !important; opacity:1 !important; font-weight:700 !important;">{html.escape(b)}</td>
            </tr>
            """
        )
    return "\n".join(rows)


def export_html_report(
    solver: GraphColoringSolver,
    found: bool,
    color_count: int,
    algorithm: str,
    graph_image: Path,
    tree_image: Path,
    output_path: Path,
) -> None:
    graph_b64 = image_to_base64(graph_image)
    tree_b64 = image_to_base64(tree_image)

    valid, errors = solver.validate_solution()
    conclusion_ok = found and valid
    status_label = "Hợp lệ" if conclusion_ok else "Cần kiểm tra"
    status_title = "Tìm được nghiệm hợp lệ" if conclusion_ok else "Chưa có nghiệm hợp lệ"
    status_desc = (
        "Tất cả khu vực kề nhau đều được tô khác màu."
        if conclusion_ok
        else "Có khu vực chưa tô hoặc tồn tại cạnh bị trùng màu."
    )
    status_class = "is-success" if conclusion_ok else "is-warning"

    if errors:
        validation_html = "<ul>" + "".join(f"<li>{html.escape(e)}</li>" for e in errors) + "</ul>"
    else:
        validation_html = "<p class='success-text'>Không phát hiện cạnh nào bị trùng màu.</p>"

    metrics = [
        ("Khu vực", len(solver.regions)),
        ("Cạnh kề", len(solver.edges)),
        ("Số màu", color_count),
        ("Lần thử", solver.try_count),
        ("Vi phạm", solver.violation_count),
        ("Quay lui", solver.backtrack_count),
        ("Node cây", len(solver.tree_nodes)),
        ("Chiến lược", html.escape(algorithm.upper())),
    ]
    metrics_html = "".join(
        f"""
        <div class="metric-card">
            <div class="metric-value">{value}</div>
            <div class="metric-label">{html.escape(label)}</div>
        </div>
        """
        for label, value in metrics
    )

    palette_html = "".join(
        f"""
        <span class="color-chip">
            <span class="color-dot" style="background:{color_hex};"></span>
            {html.escape(name)}
        </span>
        """
        for name, color_hex in COLOR_PALETTES[color_count]
    )

    logs_html = "<br>".join(html.escape(line) for line in solver.logs)
    algorithm_label = html.escape(ALGORITHM_LABELS.get(algorithm, algorithm))

    html_content = f"""<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Báo cáo tô màu đồ thị Thủ Đức</title>
    <style>
        :root {{
            --bg: #f1f5f9;
            --surface: #ffffff;
            --text: #0f172a;
            --muted: #64748b;
            --line: #e2e8f0;
            --primary: #2563eb;
            --primary-dark: #1e3a8a;
            --accent: #7c3aed;
            --success: #16a34a;
            --warning: #dc2626;
            --shadow: 0 18px 45px rgba(15, 23, 42, 0.10);
        }}
        * {{ box-sizing: border-box; }}
        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background:
                radial-gradient(circle at top left, rgba(37,99,235,0.18), transparent 34rem),
                radial-gradient(circle at top right, rgba(124,58,237,0.16), transparent 30rem),
                var(--bg);
            color: var(--text);
        }}
        .hero {{
            max-width: 1180px;
            margin: 26px auto 0;
            padding: 26px;
            border-radius: 26px;
            background: linear-gradient(135deg, #1d4ed8, #7c3aed);
            color: #fff;
            box-shadow: var(--shadow);
        }}
        .hero-top {{
            display: flex;
            justify-content: space-between;
            gap: 18px;
            align-items: flex-start;
            flex-wrap: wrap;
        }}
        .eyebrow {{
            display: inline-flex;
            align-items: center;
            gap: 8px;
            padding: 6px 12px;
            border-radius: 999px;
            background: rgba(255,255,255,0.16);
            font-size: 13px;
            font-weight: 700;
        }}
        h1 {{ margin: 14px 0 8px; font-size: 34px; line-height: 1.18; }}
        .hero p {{ margin: 0; opacity: .92; line-height: 1.55; }}
        .status-pill {{
            padding: 10px 14px;
            border-radius: 999px;
            background: rgba(255,255,255,0.18);
            font-weight: 800;
            white-space: nowrap;
        }}
        .container {{
            max-width: 1180px;
            margin: 22px auto;
            padding: 0 16px 40px;
        }}
        .panel {{
            background: rgba(255,255,255,0.95);
            border: 1px solid rgba(226,232,240,0.95);
            border-radius: 22px;
            padding: 20px;
            box-shadow: var(--shadow);
            margin-bottom: 18px;
        }}
        .panel h2 {{ margin: 0 0 10px; font-size: 22px; }}
        .note {{ color: var(--muted); margin: 0 0 14px; line-height: 1.55; }}
        .status-box {{
            display: grid;
            grid-template-columns: 1fr;
            gap: 8px;
            border-left: 7px solid var(--success);
            background: #dcfce7;
            color: #14532d;
        }}
        .status-box.is-warning {{
            border-left-color: var(--warning);
            background: #fee2e2;
            color: #7f1d1d;
        }}
        .status-title {{ font-size: 22px; font-weight: 900; }}
        .metric-grid {{
            display: grid;
            grid-template-columns: repeat(4, minmax(0, 1fr));
            gap: 14px;
            margin: 16px 0;
        }}
        .metric-card {{
            background: #fff;
            border: 1px solid var(--line);
            border-radius: 18px;
            padding: 16px;
        }}
        .metric-value {{
            font-size: 28px;
            font-weight: 900;
            color: var(--primary);
            line-height: 1;
        }}
        .metric-label {{
            margin-top: 8px;
            color: var(--muted);
            font-size: 13px;
            font-weight: 700;
        }}
        .chip-row {{ display: flex; flex-wrap: wrap; gap: 8px; margin-top: 12px; }}
        .color-chip {{
            display: inline-flex;
            align-items: center;
            gap: 8px;
            border: 1px solid var(--line);
            border-radius: 999px;
            padding: 6px 12px;
            background: #fff;
            color: var(--text);
            font-weight: 700;
            font-size: 13px;
        }}
        .color-dot {{
            width: 16px;
            height: 16px;
            display: inline-block;
            border-radius: 999px;
            border: 1px solid rgba(15,23,42,.25);
        }}
        .image-grid {{
            display: grid;
            grid-template-columns: 1fr;
            gap: 18px;
        }}
        .image-card img {{
            width: 100%;
            border-radius: 16px;
            border: 1px solid var(--line);
            background: #fff;
            display: block;
        }}
        table {{
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            overflow: hidden;
            border: 1px solid var(--line);
            border-radius: 14px;
            font-size: 14px;
        }}
        th, td {{
            text-align: left;
            border-bottom: 1px solid var(--line);
            padding: 11px 12px;
            vertical-align: top;
        }}
        th {{ background: #eff6ff !important; color: #0f172a !important; font-weight: 900 !important; }}
        tr:last-child td {{ border-bottom: none; }}
        .log {{
            background: #0f172a;
            color: #e5e7eb;
            border-radius: 16px;
            padding: 16px;
            font-family: Consolas, Monaco, monospace;
            font-size: 13px;
            max-height: 460px;
            overflow: auto;
            line-height: 1.65;
        }}
        .success-text {{ color: var(--success); font-weight: 800; }}
        @media (max-width: 850px) {{
            .metric-grid {{ grid-template-columns: repeat(2, minmax(0, 1fr)); }}
            h1 {{ font-size: 28px; }}
        }}
        @media (max-width: 520px) {{
            .metric-grid {{ grid-template-columns: 1fr; }}
            .hero {{ border-radius: 0; margin-top: 0; }}
            .container {{ padding: 0 10px 28px; }}
        }}
    </style>
</head>
<body>
    <section class="hero">
        <div class="hero-top">
            <div>
                <span class="eyebrow">Graph Coloring • Thủ Đức</span>
                <h1>Báo cáo tô màu đồ thị khu vực Thủ Đức</h1>
                <p>Thuật toán: <strong>{algorithm_label}</strong>. Điều kiện: hai khu vực kề nhau không được trùng màu.</p>
            </div>
            <div class="status-pill">{html.escape(status_label)}</div>
        </div>
    </section>

    <main class="container">
        <section class="panel status-box {status_class}">
            <div class="status-title">{html.escape(status_title)}</div>
            <div>{html.escape(status_desc)}</div>
            {validation_html}
        </section>

        <section class="panel">
            <h2>Thống kê lần chạy</h2>
            <p class="note">Các chỉ số giúp so sánh độ hiệu quả giữa các thuật toán.</p>
            <div class="metric-grid">
                {metrics_html}
            </div>
            <div class="chip-row">
                {palette_html}
            </div>
        </section>

        <section class="image-grid">
            <div class="panel image-card">
                <h2>Đồ thị sau khi tô màu</h2>
                <p class="note">Mỗi node là một khu vực, mỗi cạnh biểu diễn quan hệ giáp ranh.</p>
                <img src="data:image/png;base64,{graph_b64}" alt="Đồ thị tô màu">
            </div>

            <div class="panel image-card">
                <h2>Cây Backtracking / quá trình thử màu</h2>
                <p class="note">Xanh lá: hợp lệ, đỏ: vi phạm, xám: quay lui, xanh dương: nghiệm.</p>
                <img src="data:image/png;base64,{tree_b64}" alt="Cây backtracking">
            </div>
        </section>

        <section class="panel">
            <h2>Bảng kết quả tô màu</h2>
            <p class="note">Bảng liệt kê màu được gán, số vùng kề và danh sách vùng kề của từng khu vực.</p>
            <table>
                <thead>
                    <tr>
                        <th>Khu vực</th>
                        <th>Màu</th>
                        <th>Số vùng kề</th>
                        <th>Các vùng kề</th>
                    </tr>
                </thead>
                <tbody>
                    {result_table_html(solver)}
                </tbody>
            </table>
        </section>

        <section class="panel">
            <h2>Dữ liệu cạnh</h2>
            <table>
                <thead>
                    <tr>
                        <th>#</th>
                        <th>Khu vực A</th>
                        <th>Khu vực B</th>
                    </tr>
                </thead>
                <tbody>
                    {edges_table_html(solver)}
                </tbody>
            </table>
        </section>

        <section class="panel">
            <h2>Log thuật toán</h2>
            <div class="log">{logs_html}</div>
        </section>
    </main>
</body>
</html>
"""
    output_path.write_text(html_content, encoding="utf-8")




# ============================================================
# 7. HÀM HIỂN THỊ KẾT QUẢ RÕ CHỮ TRONG JUPYTER/COLAB
# ============================================================

def build_summary_html(solver, found, color_count, algorithm, graph_image, tree_image, csv_file, html_file):
    result_text = "TÌM ĐƯỢC NGHIỆM" if found else "KHÔNG CÓ NGHIỆM"
    result_bg = "#dcfce7" if found else "#fee2e2"
    result_color = "#14532d" if found else "#7f1d1d"
    result_icon = "✅" if found else "⚠️"

    metric_items = [
        ("Thuật toán", ALGORITHM_LABELS[algorithm]),
        ("Số màu", color_count),
        ("Số khu vực", len(solver.regions)),
        ("Số cạnh", len(solver.edges)),
        ("Lần thử màu", solver.try_count),
        ("Nhánh vi phạm", solver.violation_count),
        ("Lần quay lui", solver.backtrack_count),
        ("Node cây", len(solver.tree_nodes)),
    ]

    metric_cards = "".join(
        f"""
        <div style="
            background:#ffffff !important;
            border:1px solid #e2e8f0;
            border-radius:16px;
            padding:14px 16px;
            box-shadow:0 10px 22px rgba(15,23,42,0.06);
        ">
            <div style="color:#64748b !important; font-size:12px; font-weight:800; text-transform:uppercase; letter-spacing:.04em;">
                {html.escape(str(label))}
            </div>
            <div style="color:#1d4ed8 !important; font-size:22px; font-weight:900; margin-top:8px; line-height:1.2;">
                {html.escape(str(value))}
            </div>
        </div>
        """
        for label, value in metric_items
    )

    assignment_rows = []
    for index, region in enumerate(solver.regions, start=1):
        color_name = solver.assignment.get(region, "Chưa tô")
        color_hex = get_color_hex(color_name, color_count) if color_name != "Chưa tô" else "#f8fafc"
        assignment_rows.append(f"""
        <tr style="background:#ffffff !important; color:#111827 !important;">
            <td style="padding:10px 12px; border-bottom:1px solid #e2e8f0; color:#64748b !important; font-weight:800;">{index}</td>
            <td style="padding:10px 12px; border-bottom:1px solid #e2e8f0; color:#111827 !important; font-weight:900;">{html.escape(region)}</td>
            <td style="padding:10px 12px; border-bottom:1px solid #e2e8f0; color:#111827 !important; font-weight:900;">
                <span style="display:inline-flex; align-items:center; gap:8px; border:1px solid #cbd5e1; border-radius:999px; padding:5px 11px; background:#ffffff !important;">
                    <span style="display:inline-block; width:14px; height:14px; border-radius:999px; border:1px solid rgba(15,23,42,.25); background:{color_hex};"></span>
                    {html.escape(color_name)}
                </span>
            </td>
            <td style="padding:10px 12px; border-bottom:1px solid #e2e8f0; color:#334155 !important; font-weight:700;">{solver.degree(region)}</td>
        </tr>
        """)

    return f"""
    <div style="
        background:linear-gradient(180deg,#f8fafc,#eef2ff) !important;
        color:#0f172a !important;
        border:1px solid #dbeafe;
        border-radius:22px;
        padding:20px;
        margin:14px 0;
        font-family:Arial, Helvetica, sans-serif;
        box-shadow:0 18px 45px rgba(15,23,42,0.10);
        opacity:1 !important;
    ">
        <div style="
            background:linear-gradient(135deg,#1d4ed8,#7c3aed) !important;
            color:white !important;
            border-radius:20px;
            padding:22px;
            margin-bottom:16px;
        ">
            <div style="font-size:13px; font-weight:900; opacity:.9; letter-spacing:.04em; text-transform:uppercase;">
                Graph Coloring Dashboard
            </div>
            <h2 style="margin:8px 0 6px; color:white !important; font-size:28px; font-weight:900;">
                Tô màu đồ thị Thủ Đức
            </h2>
            <p style="margin:0; color:white !important; opacity:.92;">
                Giao diện tóm tắt kết quả chạy thuật toán và các file đã xuất.
            </p>
        </div>

        <div style="
            background:{result_bg} !important;
            color:{result_color} !important;
            border-left:7px solid {'#22c55e' if found else '#ef4444'};
            padding:14px 16px;
            border-radius:14px;
            margin-bottom:16px;
            font-weight:900 !important;
            font-size:17px;
        ">
            {result_icon} Kết quả: {result_text}
        </div>

        <div style="display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; margin-bottom:18px;">
            {metric_cards}
        </div>

        <h3 style="color:#0f172a !important; font-weight:900 !important; margin:14px 0 10px;">Bảng tô màu</h3>
        <div style="overflow:auto; border-radius:16px; border:1px solid #e2e8f0; background:#ffffff !important;">
            <table style="width:100%; border-collapse:collapse; background:#ffffff !important; color:#111827 !important;">
                <tr style="background:#dbeafe !important;">
                    <th style="text-align:left; padding:11px 12px; background:#dbeafe !important; color:#0f172a !important; font-weight:900;">#</th>
                    <th style="text-align:left; padding:11px 12px; background:#dbeafe !important; color:#0f172a !important; font-weight:900;">Khu vực</th>
                    <th style="text-align:left; padding:11px 12px; background:#dbeafe !important; color:#0f172a !important; font-weight:900;">Màu</th>
                    <th style="text-align:left; padding:11px 12px; background:#dbeafe !important; color:#0f172a !important; font-weight:900;">Số vùng kề</th>
                </tr>
                {''.join(assignment_rows)}
            </table>
        </div>

        <h3 style="color:#0f172a !important; font-weight:900 !important; margin:16px 0 8px;">File đã xuất</h3>
        <div style="display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:10px; color:#111827 !important; font-weight:800;">
            <div style="background:#ffffff !important; border:1px solid #e2e8f0; border-radius:14px; padding:12px;">Ảnh đồ thị<br><code style="color:#1d4ed8 !important; font-weight:900;">{html.escape(str(graph_image))}</code></div>
            <div style="background:#ffffff !important; border:1px solid #e2e8f0; border-radius:14px; padding:12px;">Ảnh cây<br><code style="color:#1d4ed8 !important; font-weight:900;">{html.escape(str(tree_image))}</code></div>
            <div style="background:#ffffff !important; border:1px solid #e2e8f0; border-radius:14px; padding:12px;">CSV<br><code style="color:#1d4ed8 !important; font-weight:900;">{html.escape(str(csv_file))}</code></div>
            <div style="background:#ffffff !important; border:1px solid #e2e8f0; border-radius:14px; padding:12px;">HTML report<br><code style="color:#1d4ed8 !important; font-weight:900;">{html.escape(str(html_file))}</code></div>
        </div>
    </div>
    """



def print_cli_summary(solver, found, color_count, algorithm, graph_image, tree_image, csv_file, html_file):
    print("=" * 78)
    print("TÔ MÀU ĐỒ THỊ THỦ ĐỨC - INTERACTIVE PRO")
    print("=" * 78)
    print("Kết quả:", "TÌM ĐƯỢC NGHIỆM" if found else "KHÔNG CÓ NGHIỆM")
    print("Thuật toán:", ALGORITHM_LABELS[algorithm])
    print("Số màu:", color_count)
    print("Số khu vực:", len(solver.regions))
    print("Số cạnh:", len(solver.edges))
    print("Số lần thử màu:", solver.try_count)
    print("Số nhánh vi phạm:", solver.violation_count)
    print("Số lần quay lui:", solver.backtrack_count)
    print("Số node cây:", len(solver.tree_nodes))
    print("-" * 78)
    print("BẢNG TÔ MÀU:")
    for region in solver.regions:
        print(f"{region:22s} -> {solver.assignment.get(region, 'Chưa tô')}")
    print("-" * 78)
    print("FILE ĐÃ XUẤT:")
    print("Ảnh đồ thị:", graph_image)
    print("Ảnh cây:", tree_image)
    print("CSV:", csv_file)
    print("HTML report:", html_file)
    print("=" * 78)


# ============================================================
# 7. HÀM CHẠY CHÍNH
# ============================================================

def run_coloring(
    color_count: int = 3,
    algorithm: str = "mrv",
    edges_text: Optional[str] = None,
    max_tree_nodes: int = 120,
    output_dir: str = "ket_qua_to_mau_thu_duc_interactive",
    show_report: bool = True,
) -> RunResult:
    """
    Hàm chạy chính.

    Tham số:
    - color_count: 3, 4 hoặc 5
    - algorithm:
        normal  = Backtracking theo danh sách
        degree  = Backtracking chọn bậc lớn trước
        mrv     = Backtracking MRV + Degree
        greedy  = Greedy bậc lớn trước
    - edges_text: text cạnh. Nếu None thì dùng dữ liệu mặc định.
    - max_tree_nodes: số node tối đa hiển thị trong ảnh cây.
    - output_dir: thư mục xuất kết quả.
    - show_report: True nếu muốn hiển thị HTML ngay trong notebook.
    """
    if color_count not in COLOR_PALETTES:
        raise ValueError("color_count phải là 3, 4 hoặc 5.")

    if algorithm not in ALGORITHM_LABELS:
        raise ValueError(f"algorithm phải thuộc: {', '.join(ALGORITHM_LABELS)}")

    if edges_text is None:
        regions = list(DEFAULT_REGIONS)
        edges = list(DEFAULT_EDGES)
    else:
        regions, edges = parse_edges_text(edges_text)

    color_names = [name for name, _ in COLOR_PALETTES[color_count]]

    solver = GraphColoringSolver(
        regions=regions,
        edges=edges,
        color_names=color_names,
        algorithm=algorithm,
    )

    found = solver.solve()

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    graph_image = out_dir / "do_thi_to_mau_thu_duc.png"
    tree_image = out_dir / "cay_backtracking_to_mau_thu_duc.png"
    csv_file = out_dir / "ket_qua_to_mau_thu_duc.csv"
    html_file = out_dir / "bao_cao_to_mau_thu_duc.html"

    draw_colored_graph(solver, color_count, graph_image)
    draw_backtracking_tree(solver.tree_nodes, tree_image, max_tree_nodes)
    export_csv(solver, csv_file)
    export_html_report(
        solver=solver,
        found=found,
        color_count=color_count,
        algorithm=algorithm,
        graph_image=graph_image,
        tree_image=tree_image,
        output_path=html_file,
    )

    stats = {
        "regions": len(solver.regions),
        "edges": len(solver.edges),
        "colors": color_count,
        "tries": solver.try_count,
        "violations": solver.violation_count,
        "backtracks": solver.backtrack_count,
        "tree_nodes": len(solver.tree_nodes),
    }

    if show_report:
        try:
            from IPython.display import HTML, display
            display(HTML(build_summary_html(
                solver=solver,
                found=found,
                color_count=color_count,
                algorithm=algorithm,
                graph_image=graph_image,
                tree_image=tree_image,
                csv_file=csv_file,
                html_file=html_file,
            )))
        except Exception:
            print_cli_summary(
                solver=solver,
                found=found,
                color_count=color_count,
                algorithm=algorithm,
                graph_image=graph_image,
                tree_image=tree_image,
                csv_file=csv_file,
                html_file=html_file,
            )
    else:
        print_cli_summary(
            solver=solver,
            found=found,
            color_count=color_count,
            algorithm=algorithm,
            graph_image=graph_image,
            tree_image=tree_image,
            csv_file=csv_file,
            html_file=html_file,
        )

    if show_report:
        try:
            from IPython.display import HTML, display
            report_html = html_file.read_text(encoding="utf-8")
            display(HTML(report_html))
        except Exception:
            print("Không hiển thị HTML trực tiếp được. Hãy mở file:", html_file)

    return RunResult(
        found=found,
        assignment=dict(solver.assignment),
        tree_nodes=list(solver.tree_nodes),
        logs=list(solver.logs),
        output_dir=out_dir,
        graph_image=graph_image,
        tree_image=tree_image,
        csv_file=csv_file,
        html_file=html_file,
        stats=stats,
    )


# ============================================================
# 8. GIAO DIỆN JUPYTER BẰNG IPYWIDGETS
# ============================================================

def launch_widget_app() -> None:
    """
    Mở giao diện chọn trực tiếp trong Jupyter/Colab.
    """
    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML, clear_output
    except Exception as exc:
        print("Không tải được ipywidgets.")
        print("Cài bằng lệnh: !pip install ipywidgets")
        print("Lỗi:", exc)
        print("\nChạy fallback bằng dòng lệnh:")
        print("run_coloring(color_count=3, algorithm='mrv')")
        return

    app_css = widgets.HTML(
        """
        <style>
            .td-app * { box-sizing: border-box; }
            .td-app {
                font-family: Arial, Helvetica, sans-serif;
                color: #0f172a;
            }
            .td-hero {
                padding: 22px 24px;
                border-radius: 22px;
                background: linear-gradient(135deg, #1d4ed8, #7c3aed);
                color: white !important;
                box-shadow: 0 18px 45px rgba(30, 64, 175, .24);
                margin-bottom: 14px;
            }
            .td-hero h2 {
                margin: 6px 0 6px;
                color: white !important;
                font-size: 28px;
                font-weight: 900;
            }
            .td-hero p {
                margin: 0;
                color: white !important;
                opacity: .92;
                line-height: 1.55;
            }
            .td-pill {
                display: inline-block;
                padding: 6px 12px;
                border-radius: 999px;
                background: rgba(255,255,255,.16);
                color: white !important;
                font-size: 12px;
                font-weight: 900;
                letter-spacing: .04em;
                text-transform: uppercase;
            }
            .td-card {
                background: #ffffff;
                border: 1px solid #e2e8f0;
                border-radius: 18px;
                padding: 16px;
                box-shadow: 0 10px 26px rgba(15,23,42,.07);
                margin: 8px 0;
            }
            .td-card h3 {
                margin: 0 0 8px;
                color: #0f172a !important;
                font-size: 17px;
                font-weight: 900;
            }
            .td-note {
                margin: 0;
                color: #64748b !important;
                line-height: 1.55;
                font-size: 13px;
            }
            .td-guide {
                background: #eff6ff;
                border-left: 5px solid #2563eb;
                color: #0f172a !important;
                border-radius: 14px;
                padding: 12px 14px;
                line-height: 1.55;
                font-size: 13px;
                margin: 8px 0;
            }
        </style>
        """
    )

    title = widgets.HTML(
        """
        <div class="td-app">
            <div class="td-hero">
                <span class="td-pill">Interactive Pro</span>
                <h2>Tô màu đồ thị khu vực Thủ Đức</h2>
                <p>Chọn số màu, thuật toán, chỉnh danh sách cạnh rồi chạy. Kết quả sẽ xuất ra ảnh, CSV và báo cáo HTML.</p>
            </div>
        </div>
        """
    )

    color_dropdown = widgets.Dropdown(
        options=[
            ("3 màu: Đỏ, Xanh, Vàng", 3),
            ("4 màu: thêm Tím", 4),
            ("5 màu: thêm Xanh lá", 5),
        ],
        value=3,
        description="Số màu",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    algorithm_dropdown = widgets.Dropdown(
        options=[
            ("Backtracking - MRV + Degree", "mrv"),
            ("Backtracking - Bậc lớn trước", "degree"),
            ("Backtracking - Theo danh sách", "normal"),
            ("Greedy - Bậc lớn trước", "greedy"),
        ],
        value="mrv",
        description="Thuật toán",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    max_tree_slider = widgets.IntSlider(
        value=120,
        min=20,
        max=300,
        step=10,
        description="Node cây",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
        continuous_update=False,
    )

    output_text = widgets.Text(
        value="ket_qua_to_mau_thu_duc_interactive",
        description="Thư mục",
        placeholder="Tên thư mục xuất kết quả",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="100%"),
    )

    edges_area = widgets.Textarea(
        value=edges_to_text(DEFAULT_EDGES),
        placeholder="Ví dụ: Linh Xuân - Bình Chiểu",
        layout=widgets.Layout(width="100%", height="310px"),
    )

    settings_title = widgets.HTML(
        """
        <div class="td-app"><div class="td-card">
            <h3>Thông số chạy</h3>
            <p class="td-note">MRV + Degree thường cho kết quả gọn và ít thử màu hơn.</p>
        </div></div>
        """
    )

    data_title = widgets.HTML(
        """
        <div class="td-app"><div class="td-card">
            <h3>Dữ liệu cạnh</h3>
            <p class="td-note">Mỗi dòng là một cặp khu vực giáp ranh. Có thể dùng dấu <b>-</b> hoặc dấu phẩy.</p>
        </div></div>
        """
    )

    guide = widgets.HTML(
        """
        <div class="td-app">
            <div class="td-guide">
                <b>Ví dụ nhập cạnh:</b> <code>Linh Xuân - Bình Chiểu</code><br>
                Không nhập cạnh trùng, không nhập khu vực kề với chính nó.
            </div>
        </div>
        """
    )

    run_button = widgets.Button(
        description="Chạy tô màu",
        button_style="success",
        icon="play",
        tooltip="Chạy thuật toán và xuất kết quả",
        layout=widgets.Layout(width="170px", height="42px"),
    )

    reset_button = widgets.Button(
        description="Reset dữ liệu",
        button_style="warning",
        icon="refresh",
        tooltip="Khôi phục dữ liệu mặc định",
        layout=widgets.Layout(width="170px", height="42px"),
    )

    output = widgets.Output(
        layout=widgets.Layout(
            width="100%",
            border="1px solid #e2e8f0",
            border_radius="16px",
            padding="10px",
        )
    )

    def show_message(message: str, kind: str = "info") -> None:
        palette = {
            "info": ("#eff6ff", "#1d4ed8"),
            "success": ("#dcfce7", "#166534"),
            "error": ("#fee2e2", "#991b1b"),
        }
        bg, fg = palette.get(kind, palette["info"])
        display(HTML(f"""
        <div style="
            background:{bg};
            color:{fg};
            border-radius:14px;
            padding:12px 14px;
            font-family:Arial, Helvetica, sans-serif;
            font-weight:800;
            margin-bottom:10px;
        ">{html.escape(message)}</div>
        """))

    def on_reset_clicked(_button):
        edges_area.value = edges_to_text(DEFAULT_EDGES)
        color_dropdown.value = 3
        algorithm_dropdown.value = "mrv"
        max_tree_slider.value = 120
        output_text.value = "ket_qua_to_mau_thu_duc_interactive"
        with output:
            clear_output()
            show_message("Đã reset về dữ liệu mặc định.", "success")

    def on_run_clicked(_button):
        with output:
            clear_output()
            try:
                show_message("Đang chạy thuật toán và tạo báo cáo...", "info")
                run_coloring(
                    color_count=color_dropdown.value,
                    algorithm=algorithm_dropdown.value,
                    edges_text=edges_area.value,
                    max_tree_nodes=max_tree_slider.value,
                    output_dir=output_text.value,
                    show_report=True,
                )
            except Exception as exc:
                clear_output()
                show_message("Lỗi khi chạy chương trình. Kiểm tra lại dữ liệu cạnh hoặc tham số nhập.", "error")
                print(exc)

    run_button.on_click(on_run_clicked)
    reset_button.on_click(on_reset_clicked)

    left_panel = widgets.VBox(
        [settings_title, color_dropdown, algorithm_dropdown, max_tree_slider, output_text, widgets.HBox([run_button, reset_button])],
        layout=widgets.Layout(
            width="38%",
            padding="14px",
            border="1px solid #e2e8f0",
            border_radius="18px",
            box_shadow="0 10px 26px rgba(15,23,42,.07)",
        ),
    )

    right_panel = widgets.VBox(
        [data_title, guide, edges_area],
        layout=widgets.Layout(
            width="62%",
            padding="14px",
            border="1px solid #e2e8f0",
            border_radius="18px",
            box_shadow="0 10px 26px rgba(15,23,42,.07)",
        ),
    )

    controls = widgets.VBox(
        [
            app_css,
            title,
            widgets.HBox([left_panel, right_panel], layout=widgets.Layout(width="100%", gap="14px", align_items="stretch")),
            widgets.HTML("<div style='height:12px'></div>"),
            output,
        ],
        layout=widgets.Layout(width="100%"),
    )

    display(controls)



# ============================================================
# 9. CLI
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Tô màu đồ thị Thủ Đức - bản Interactive Pro.",
    )
    parser.add_argument(
        "--mode",
        choices=["auto", "widget", "cli"],
        default="auto",
        help="auto: tự chọn, widget: giao diện Jupyter, cli: chạy dòng lệnh.",
    )
    parser.add_argument(
        "--colors",
        type=int,
        choices=[3, 4, 5],
        default=3,
        help="Số màu.",
    )
    parser.add_argument(
        "--algorithm",
        choices=["normal", "degree", "mrv", "greedy"],
        default="mrv",
        help="Thuật toán.",
    )
    parser.add_argument(
        "--max-tree-nodes",
        type=int,
        default=120,
        help="Số node tối đa hiển thị trong cây.",
    )
    parser.add_argument(
        "--output-dir",
        default="ket_qua_to_mau_thu_duc_interactive",
        help="Thư mục xuất kết quả.",
    )
    args, unknown = parser.parse_known_args()
    return args


def running_inside_ipython() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


def main() -> None:
    args = parse_args()

    if args.mode == "widget":
        launch_widget_app()
        return

    if args.mode == "cli":
        run_coloring(
            color_count=args.colors,
            algorithm=args.algorithm,
            max_tree_nodes=args.max_tree_nodes,
            output_dir=args.output_dir,
            show_report=running_inside_ipython(),
        )
        return

    # auto
    if running_inside_ipython():
        launch_widget_app()
    else:
        run_coloring(
            color_count=args.colors,
            algorithm=args.algorithm,
            max_tree_nodes=args.max_tree_nodes,
            output_dir=args.output_dir,
            show_report=False,
        )


if __name__ == "__main__":
    main()
